# 가격 책정 파이프라인 데모 — 검색 → 가격 범위 산정

**이 노트북은 새 로직이 아니다.** 이미 프로덕션에 구현·머지된 `AI/app/pricing/`
(comps 수집 → 가격 산정)을 그대로 불러와서 한 흐름으로 보여주는 시연용 노트북이다.
평가 데모에서 "검색부터 가격 범위를 잡는 것까지"를 눈으로 보여주기 위한 용도.

## 파이프라인

```
상품명/브랜드/카테고리
        │
        ▼
  build_query()          — app/pricing/comps.py
        │
        ▼
  comps_source.fetch()   — mock(기본) 또는 llm_web_search(GMS 웹검색)
        │
        ▼
  pricer.price()         — 중앙값 트림 + 상태계수 (app/pricing/pricer.py)
        │
        ▼
  가격 range + 신뢰도 + 근거 문장
```

설계 근거: `.claude/plans/ai/CH003-pricing-service-prototype.md`,
`CH004-naver-retail-price.md`. 실제 서비스(`app/pricing/service.py`)는 이 위에 job
큐·재시도·신상품가 조회가 더 붙지만, 이 노트북은 핵심 흐름(검색 → 가격 범위)만 보여준다.

## 0. 준비 — 프로덕션 모듈 import

노트북 위치(`validation/PRICE/pricing-model-check/`)에서 `AI/` 루트를 `sys.path`에 넣어야
`import app...`이 해석된다(pytest는 `pyproject.toml`의 `pythonpath=["."]`가 대신 해 주지만
노트북은 직접 잡아야 한다).

In [ ]:
import os
import sys
from pathlib import Path

# validation/PRICE/pricing-model-check/ -> PRICE/ -> validation/ -> AI/
_AI_ROOT = Path.cwd().resolve().parents[2]
if str(_AI_ROOT) not in sys.path:
    sys.path.insert(0, str(_AI_ROOT))

# app.core.config.Settings는 env_file=".env"를 **현재 작업 디렉터리 기준 상대경로**로 연다.
# 노트북 커널의 cwd는 이 파일이 있는 pricing-model-check/라서, sys.path만 잡고 cwd는 그대로
# 두면 AI/.env를 못 찾아 GMS_API_KEY 등이 항상 빈 값으로 로딩된다. import 전에 cwd 자체를
# AI/로 옮겨야 한다 — 한 번 import된 뒤에는 Settings 인스턴스가 캐시되므로, 이미 mock으로
# import를 마친 커널에서 뒤늦게 cwd만 바꿔도 반영되지 않는다(커널 재시작 필요).
os.chdir(_AI_ROOT)

from app.pricing.comps import build_comps_source, build_query  # noqa: E402
from app.pricing.pricer import build_pricer  # noqa: E402
from app.pricing.registry import comps_config  # noqa: E402
from app.pricing.schemas import ConditionGrade, GarmentCategory  # noqa: E402

print(f"AI 루트(cwd로 전환됨): {_AI_ROOT}")

## 1. 백엔드 선택

`SEARCH_BACKEND`가 `"mock"`(기본)이면 네트워크·API 키 없이 결정적 매물로 전체 흐름을
바로 확인할 수 있다 — **이 노트북은 mock 기준으로 이미 실행·검증해 뒀다.**

**실제 GMS 웹검색으로 라이브 데모를 하려면**: `AI/.env`에 `GMS_API_KEY`를 채우고 아래
`SEARCH_BACKEND`를 `"llm_web_search"`로 바꾼 뒤 **사용자가 직접** 이 셀부터 재실행한다.
GMS는 팀 공용 유료 쿼터라 Claude는 이 백엔드로 실행하지 않는다(`AI/CLAUDE.md` 하드 룰).
실제 검색은 7~15초 정도 걸린다(CH002 실측).

In [ ]:
# "mock"(기본, 네트워크 없음) 또는 "llm_web_search"(실제 GMS 검색 — 사용자가 직접 전환)
SEARCH_BACKEND = "mock"

config = comps_config(SEARCH_BACKEND)
comps_source = build_comps_source(SEARCH_BACKEND)
pricer = build_pricer(min_comps=config["min_comps_for_pricing"])

print(f"comps 백엔드   : {comps_source.key}")
print(f"pricer 백엔드  : {pricer.key}")
print(f"최소 표본 기준 : {config['min_comps_for_pricing']}건")

## 2. 검색 — 상품 정보로 쿼리를 만들고 comps를 모은다

`build_query()`는 상품명+브랜드+카테고리를 중복 없이 한 줄로 합친다(`comps.py`). 여기 값을
바꾸면 다른 상품으로 데모할 수 있다.

In [ ]:
# 데모 상품 — 자유롭게 바꿔서 시연 가능
PRODUCT_NAME = "리바이스 501 빅E 데님"
BRAND = "리바이스"
CATEGORY = GarmentCategory.bottom_pants
CONDITION_SCORE = 78.0  # 품질검수 결과라고 가정한 값(0~100)
CONDITION_GRADE = ConditionGrade.a

query = build_query(PRODUCT_NAME, BRAND, CATEGORY)
print(f"검색 쿼리: {query!r}\n")

fetched = comps_source.fetch(query)
print(f"수집된 매물 {len(fetched.items)}건")
for item in fetched.items:
    used = "중고" if item.is_used else ("신품" if item.is_used is False else "불명")
    print(f"  {item.price:>10,}원  {item.title:<30} [{item.source}/{used}]")

if fetched.notes:
    print("\n안내:")
    for note in fetched.notes:
        print(f"  - {note}")

## 3. 가격 범위 산정 — 중앙값 트림 + 상태계수

`RuleBaselinePricer.price()`(대조군/기본값)가 하는 일: 신품 제외 → 표본 5건 이상이면
최고·최저 1건씩 트림 → 중앙값에 상태계수를 곱함 → 신뢰도에 따라 range 폭을 정함.
계수·경계는 아직 근거 없는 임의값이다(실판매가 정답셋으로 캘리브레이션 전) — 데모에서도
이 한계는 그대로 언급하는 게 정직하다.

In [ ]:
outcome = pricer.price(
    fetched.items,
    condition_score=CONDITION_SCORE,
    condition_grade=CONDITION_GRADE,
)

print(f"가격 산정 여부: {outcome.priced}  (사유: {outcome.reason_code.value})")

if outcome.priced:
    r = outcome.price_range
    print(f"\n제안 가격대   : {r.min:,}원 ~ {r.max:,}원  (중앙값 {r.mid:,}원)")
    print(f"신뢰도        : {outcome.confidence:.2f} ({outcome.confidence_level.value})")
    print(f"상태 계수     : {outcome.coefficient:.2f}  (등급 {CONDITION_GRADE.value}, "
          f"점수 {CONDITION_SCORE:g})")
    if outcome.comps_stats:
        s = outcome.comps_stats
        print(f"비교 매물     : {s.count}건 (중앙값 {s.median:,}원, "
              f"p25 {s.p25:,}원, p75 {s.p75:,}원, 중고 확인 비율 {s.used_ratio:.0%})")
    print("\n근거:")
    for line in outcome.rationale:
        print(f"  - {line}")
else:
    print("\n근거:")
    for line in outcome.rationale:
        print(f"  - {line}")

## 4. (선택) 신상품가 병행 조회 — 네이버 쇼핑 vs GMS

CH002가 처음엔 네이버 쇼핑 API를 **중고 시세** 후보로 봤는데, 실측해 보니 중고 비율이
5%뿐이었다(대부분 신품). 그래서 지금 프로덕션은 네이버를 중고 comps가 아니라
**신상품(정가) 앵커**로 재배치했다(`app/pricing/retail.py`의 `NaverRetailSource`, CH004).
comps(GMS 웹검색, 중고)와 retail(네이버, 신상품)은 완전히 분리된 별도 조회다 — 하나가
다른 하나를 대체하지 않는다.

`RETAIL_BACKEND`도 위 `SEARCH_BACKEND`와 같은 규칙: `"mock"`(기본, 네트워크 없음)으로
바로 실행 가능하고, 실제 네이버 API로 라이브 데모를 하려면 `AI/.env`에
`NAVER_CLIENT_ID`/`NAVER_CLIENT_SECRET`을 채운 뒤 **사용자가 직접** `"naver"`로 바꿔서
재실행한다.

In [ ]:
from app.pricing.retail import build_retail_source

# "mock"(기본, 네트워크 없음) 또는 "naver"(실제 네이버 쇼핑 API — 사용자가 직접 전환)
RETAIL_BACKEND = "mock"

retail_source = build_retail_source(RETAIL_BACKEND)
print(f"retail 백엔드: {retail_source.key}")

retail_fetched = retail_source.fetch(query)
print(f"\n조회된 신상품 {len(retail_fetched.items)}건")
for item in retail_fetched.items:
    print(f"  {item.price:>10,}원  {item.title:<30} [{item.source}]")

if retail_fetched.notes:
    print("\n안내:")
    for note in retail_fetched.notes:
        print(f"  - {note}")

### comps(중고) vs retail(신상품) — 신품 정가를 반영하면 근거 문장이 어떻게 바뀌는가

`pricer.price()`는 `retail_price`(신품 정가 참고값)를 받으면 근거 문장에 할인율을 한 줄
더 붙인다(`pricer.py`의 `_rationale()`). 프로덕션은 아직 이 값을 요청에서 직접 받는 정도이고
네이버 조회 결과를 자동으로 여기 흘려넣진 않는다(CH004가 "가격 계산 반영은 다음 라운드"로
명시해 둔 부분) — 여기서는 그 다음 라운드가 실제로 붙으면 어떤 모습일지 **같은 함수를 다시
불러서** 미리 보여준다.

In [ ]:
import statistics as _stats

if retail_fetched.items:
    retail_median = int(_stats.median(i.price for i in retail_fetched.items))
    outcome_with_retail = pricer.price(
        fetched.items,
        condition_score=CONDITION_SCORE,
        condition_grade=CONDITION_GRADE,
        retail_price=retail_median,
    )
    print(f"신상품 정가(중앙값): {retail_median:,}원\n")
    print("근거(신상품가 반영):")
    for line in outcome_with_retail.rationale:
        print(f"  - {line}")
else:
    print("신상품가를 못 얻어서 할인율 비교를 건너뜁니다.")

## 5. 가격 분포 시각화 (데모용)

**이 시각화는 프로덕션 코드가 아니다** — `CompsStats`에는 아직 구간별 집계 필드가 없다
(`.claude/plans/ai/CH006-comps-price-histogram.md`가 이걸 정식 필드로 추가하는 설계이고,
아직 승인 대기 중이다). 여기서는 수집된 `fetched.items`의 가격을 노트북에서 바로 그려서
"가격 구간별 매물 수 + 제안 range"를 눈으로 보여주는 데모 전용 셀이다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager


def _use_korean_font():
    """한글이 두부(□□)로 깨지지 않게 폰트를 잡는다. 후보가 없으면 조용히 넘어간다."""
    available = {f.name for f in font_manager.fontManager.ttflist}
    for candidate in ("Malgun Gothic", "AppleGothic", "NanumGothic", "Gulim"):
        if candidate in available:
            matplotlib.rcParams["font.family"] = candidate
            matplotlib.rcParams["axes.unicode_minus"] = False
            return candidate
    return None


print(f"matplotlib 한글 폰트: {_use_korean_font()}")

prices = [item.price for item in fetched.items]
bins = min(5, len(set(prices))) or 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(prices, bins=bins, color="#4C72B0", edgecolor="white")
if outcome.priced:
    r = outcome.price_range
    ax.axvline(r.min, color="gray", linestyle="--", label=f"min {r.min:,}")
    ax.axvline(r.mid, color="crimson", linestyle="-", label=f"mid {r.mid:,}")
    ax.axvline(r.max, color="gray", linestyle="--", label=f"max {r.max:,}")
    ax.legend()
ax.set_xlabel("가격(원)")
ax.set_ylabel("매물 수")
ax.set_title(f"'{query}' 가격 분포 + 제안 range")
plt.tight_layout()
plt.show()

---

## 참고 — 데모 시 언급할 것

- 여기서 부른 함수(`build_query`/`build_comps_source`/`build_pricer`/`build_retail_source`)는
  **실제 서비스가 쓰는 그 코드 그대로**다(`app/pricing/`). 실서비스는 이 위에 job 큐·응답
  스키마 조립이 더 붙어 있을 뿐 산정 로직 자체는 동일하다.
- 기본 `SEARCH_BACKEND`/`RETAIL_BACKEND` = `"mock"`은 상품명 해시로 만든 **가짜 매물**이다 —
  라이브로 `"llm_web_search"`/`"naver"`로 바꾸면 실제 검색 결과로 같은 파이프라인이 도는 걸
  보여줄 수 있다.
- **comps(GMS, 중고)와 retail(네이버, 신상품)은 역할이 다르다** — 택일이 아니라 상호보완이다
  (§4, LAB-47 실측: 네이버는 표본 많고 빠르지만 중고 비율 5%, GMS는 느리지만 중고 정확도 100%).
- **정직하게 밝힐 한계**: 상태계수(`GRADE_COEFFICIENTS`/`condition_coefficient`)는 아직
  실판매가로 캘리브레이션되지 않은 값이다. "왜 이 가격인지" 근거 문장은 나오지만, "이 숫자가
  실제로 정확한지"는 별도 검증(`CH002` V4)이 필요하다는 걸 데모에서도 짚어주는 게 좋다.
- 배경: `.claude/plans/ai/CH002-pricing-model-selection.md`(모델·백엔드 선정 검증),
  `CH003-pricing-service-prototype.md`(서비스 구조), `CH004-naver-retail-price.md`(신상품가
  병행 수집), Jira `S15P11A108-345`(이번 라운드 아키텍처 정리).